<a href="https://colab.research.google.com/github/nanduvarghese07/gen-ai/blob/main/DAY7/SENTIMENTAL_ANALYSIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [ ]:
vocab_size = 10000
#Keep only the top 10000 most frequent words
(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 25000
Testing samples: 25000


In [ ]:
max_length = 200

X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post'
)


In [ ]:
model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    ),

    SimpleRNN(64),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=64
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 78ms/step - accuracy: 0.5086 - loss: 0.6961 - val_accuracy: 0.5078 - val_loss: 0.6926
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 78ms/step - accuracy: 0.5177 - loss: 0.6924 - val_accuracy: 0.5038 - val_loss: 0.6969
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 25s 78ms/step - accuracy: 0.5321 - loss: 0.6847 - val_accuracy: 0.5344 - val_loss: 0.6885
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 25s 78ms/step - accuracy: 0.5746 - loss: 0.6623 - val_accuracy: 0.5604 - val_loss: 0.6662
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 24s 77ms/step - accuracy: 0.6204 - loss: 0.6101 - val_accuracy: 0.5966 - val_loss: 0.6282


In [ ]:
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("Accuracy:", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.5978 - loss: 0.6278
Accuracy: 0.597760021686554


In [ ]:
from tensorflow.keras.datasets import imdb

word_index = imdb.get_word_index()

# Shift indices by 3 because Keras reserves 0,1,2
word_index = {k: (v + 3) for k, v in word_index.items()}

word_index["<PAD>"] = 0
word_index["<START>"] = 1
word_index["<UNK>"] = 2
word_index["<UNUSED>"] = 3

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


In [ ]:
def review_to_sequence(review):

    review = review.lower().split()

    sequence = []

    for word in review:
        sequence.append(word_index.get(word, 2))  # 2 = unknown word

    return sequence

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

max_length = 200

def predict_sentiment(review):

    sequence = review_to_sequence(review)

    padded = pad_sequences(
        [sequence],
        maxlen=max_length,
        padding='post'
    )

    prediction = model.predict(padded, verbose=0)

    score = prediction[0][0]

    print("Sentiment Score:", score)

    if score > 0.5:
        print("Positive Review ")
    else:
        print("Negative Review ")

In [ ]:
predict_sentiment(
    "This movie was amazing and the acting was excellent"
)

Sentiment Score: 0.5172129
Positive Review 😊


In [ ]:
predict_sentiment(
    "i dont like this movie "
)

Sentiment Score: 0.5325405
Positive Review 😊


IMDB DATASET

In [ ]:
import pandas as pd

df = pd.read_csv("/content/IMDB Dataset.csv", encoding='latin1')

print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [ ]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [ ]:
import re

negation_words = [
    "not good", "not bad", "not great", "don't like",
    "didn't like", "never liked", "wasn't good",
    "isn't good", "no good"
]

def clean_text(text):
    text = text.lower()

    text = re.sub(r"[^a-zA-Z\s']", " ", text)

    # convert negations into single tokens
    for phrase in negation_words:
        text = text.replace(phrase, phrase.replace(" ", "_"))

    return text

| Original phrase | Converted    |
| --------------- | ------------ |
| "not good"      | "not_good"   |
| "don't like"    | "don't_like" |


eg:This movie is not good at all

after transformation :This movie is not_good at all

In [ ]:
df['review'] = df['review'].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)

Every review is converted into a list of numbers.

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len),

    LSTM(128, dropout=0.3, recurrent_dropout=0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 390s 771ms/step - accuracy: 0.5549 - loss: 0.6706 - val_accuracy: 0.5938 - val_loss: 0.6331
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 439s 766ms/step - accuracy: 0.5927 - loss: 0.6250 - val_accuracy: 0.5968 - val_loss: 0.6247
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 445s 773ms/step - accuracy: 0.7605 - loss: 0.4991 - val_accuracy: 0.8372 - val_loss: 0.4179
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 456s 799ms/step - accuracy: 0.8600 - loss: 0.3573 - val_accuracy: 0.8593 - val_loss: 0.3509
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 380s 761ms/step - accuracy: 0.9173 - loss: 0.2335 - val_accuracy: 0.8814 - val_loss: 0.3008


In [ ]:
loss, acc = model.evaluate(X_test_pad, y_test)

print("Test Accuracy:", acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 95ms/step - accuracy: 0.8865 - loss: 0.2949
Test Accuracy: 0.8865000009536743


In [ ]:
def predict_sentiment(review):

    review = clean_text(review)

    seq = tokenizer.texts_to_sequences([review])

    padded = pad_sequences(seq, maxlen=max_len, padding='post')

    prediction = model.predict(padded)[0][0]

    print("\nReview:", review)
    print("Score:", prediction)

    if prediction >= 0.5:
        print("Sentiment: Positive 😊")
    else:
        print("Sentiment: Negative 😞")

In [ ]:
predict_sentiment("This movie was absolutely amazing and I loved it")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 394ms/step

Review: this movie was absolutely amazing and i loved it
Score: 0.9611797
Sentiment: Positive 😊


In [ ]:
predict_sentiment("This movie was not good and I didn't like it at all")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step

Review: this movie was not_good and i didn't_like it at all
Score: 0.33284366
Sentiment: Negative 😞


In [ ]:
predict_sentiment("This movie is very entertaining! ")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step

Review: this movie is very entertaining  
Score: 0.9317532
Sentiment: Positive 😊
